In [1]:
import optuna

from optuna.integration import XGBoostPruningCallback

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

/Users/cinar/.pyenv/versions/3.13.14/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [3]:
def objective(trial):
    max_depth=trial.suggest_int("max_depth",3,8)
    learning_rate=trial.suggest_float("learning_rate",0.01,0.3)
    n_estimators=trial.suggest_int("n_estimators",50,300)

    model=XGBClassifier(max_depth=max_depth,learning_rate=learning_rate,n_estimators=n_estimators,
                        eval_metric="logloss",random_state=1)

    pruning_callback=XGBoostPruningCallback(trial,"validation_0-logloss")

    model.fit(X_train,y_train,eval_set=[(X_test,y_test)],callbacks=[pruning_callback],verbose=False)

    predictions=model.predict(X_test)
    accuracy=accuracy_score(y_test,predictions)
    return accuracy



In [4]:
study=optuna.create_study(direction="maximize",pruner=optuna.pruners.MedianPruner())

[I 2026-07-07 23:17:07,507] A new study created in memory with name: no-name-f13608ac-c6c5-4af2-add0-e411a1671d00


In [5]:
study.optimize(objective,n_trials=30)

/Users/cinar/.pyenv/versions/3.13.14/lib/python3.13/site-packages/xgboost/sklearn.py:835: UserWarning: `callbacks` in `fit` method is deprecated for better compatibility with scikit-learn, use `callbacks` in constructor or`set_params` instead.
  warnings.warn(
[I 2026-07-07 23:17:07,534] Trial 0 finished with value: 0.885 and parameters: {'max_depth': 3, 'learning_rate': 0.23787746385677355, 'n_estimators': 50}. Best is trial 0 with value: 0.885.
/Users/cinar/.pyenv/versions/3.13.14/lib/python3.13/site-packages/xgboost/sklearn.py:835: UserWarning: `callbacks` in `fit` method is deprecated for better compatibility with scikit-learn, use `callbacks` in constructor or`set_params` instead.
  warnings.warn(
[I 2026-07-07 23:17:07,617] Trial 1 finished with value: 0.89 and parameters: {'max_depth': 6, 'learning_rate': 0.21316281929509667, 'n_estimators': 194}. Best is trial 1 with value: 0.89.
/Users/cinar/.pyenv/versions/3.13.14/lib/python3.13/site-packages/xgboost/sklearn.py:835: UserWarni

In [6]:
len(study.trials)

30

In [7]:
pruned=len([

    t for t in study.trials 
    if t.state == optuna.trial.TrialState.PRUNED

])

In [8]:
pruned 

14

In [9]:
study.best_value

0.905

In [10]:
import optuna.visualization as vis 

In [12]:
fig=vis.plot_optimization_history(study)
fig.show()

In [13]:
fig=vis.plot_param_importances(study)
fig.show()

In [14]:
fig=vis.plot_contour(study)
fig.show()

In [15]:
fig=vis.plot_slice(study)
fig.show()

In [16]:
fig=vis.plot_parallel_coordinate(study)
fig.show()

In [17]:
study.best_value

0.905

In [18]:
for key , value in study.best_params.items():
    print(f"{key}: {value}")

max_depth: 3
learning_rate: 0.07266640498823333
n_estimators: 161
